# UE5 Heightmap + Track Prep


## Imports

In [ ]:
import os, glob, imageio, json
import numpy as np
import pandas as pd
import rasterio
from rasterio.merge import merge
from rasterio.io import MemoryFile
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.fill import fillnodata
import matplotlib.pyplot as plt
import pyproj
import gpxpy


## Heightmap

Loads USGS DEM tiles and a GPX track (for centroid / UTM zone), mosaics, reprojects to UTM, crops to a tile-aligned window, normalizes to 16-bit, and exports:
- `Heightmap.png` — ready for UE5 landscape import
- `reprojected_dem.tif` — the DEM in UTM at the same grid, for downstream rasters
- `heightmap_bounds.json` — grid origin, extent, and pixel size for alignment

In [ ]:
# --- PARAMETERS ---
input_paths = glob.glob("../data/wurl/usgs_tiles/*.tif")
gpx_path = "../data/wurl/WURL_Wasatch_Ultimate_Ridge_Linkup.gpx"
output_dir = "../data/wurl/unreal_output"
tile_size = 4033

os.makedirs(output_dir, exist_ok=True)

# --- 1. LOAD GPX & DETERMINE UTM ZONE ---
with open(gpx_path) as f:
    gpx = gpxpy.parse(f)
points = []
for track in gpx.tracks:
    for segment in track.segments:
        for pt in segment.points:
            points.append({'lat': pt.latitude, 'lon': pt.longitude, 'ele': pt.elevation or 0})
if not points:
    raise ValueError("No track points found in GPX.")
print(f"Loaded GPX: {len(points)} points")

center_lon = np.mean([p['lon'] for p in points])
center_lat = np.mean([p['lat'] for p in points])
zone = int(np.floor((center_lon + 180) / 6)) + 1
hemisphere = 6 if center_lat >= 0 else 7
target_crs = f"EPSG:326{zone:02d}" if hemisphere == 6 else f"EPSG:327{zone:02d}"
print(f"UTM zone: {zone}{'N' if hemisphere == 6 else 'S'}  ({target_crs})")

# Reproject GPX to UTM meters
proj = pyproj.Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)
coords = np.array([proj.transform(p['lon'], p['lat']) for p in points])

print(f"Track bounds (UTM m): E [{coords[:,0].min():.0f}, {coords[:,0].max():.0f}], "
      f"N [{coords[:,1].min():.0f}, {coords[:,1].max():.0f}]")

# --- 2. MOSAIC USGS TILES ---
opened = []
if len(input_paths) == 1:
    src = rasterio.open(input_paths[0])
    opened.append(src)
else:
    handles = [rasterio.open(p) for p in input_paths]
    mosaic_array, mosaic_transform = merge(handles)
    profile = handles[0].profile.copy()
    profile.update(height=mosaic_array.shape[1], width=mosaic_array.shape[2], transform=mosaic_transform)
    mem = MemoryFile()
    src = mem.open(**profile)
    src.write(mosaic_array)
    opened.extend(handles + [mem, src])

# --- 3. REPROJECT TO UTM ---
dst_crs = rasterio.crs.CRS.from_string(target_crs)
tfm, w_out, h_out = calculate_default_transform(src.crs, dst_crs, src.width, src.height, *src.bounds)
p = src.profile.copy(); p.update(crs=dst_crs, transform=tfm, width=w_out, height=h_out)
m = MemoryFile(); src_r = m.open(**p)
for i in range(1, src.count + 1):
    reproject(rasterio.band(src, i), rasterio.band(src_r, i),
              src_transform=src.transform, src_crs=src.crs,
              dst_transform=tfm, dst_crs=dst_crs, resampling=Resampling.bilinear)
opened += [m, src_r]
print(f"Reprojected to {target_crs}: {w_out}x{h_out}")

pixel_size_m = abs(src_r.transform.a)

# --- 4. FIND GPX CENTROID AND SET TILE-ALIGNED WINDOW ---
inv = ~src_r.transform
gpx_cols = inv.a * coords[:, 0] + inv.c
gpx_rows = inv.e * coords[:, 1] + inv.f
centroid_col = gpx_cols.mean()
centroid_row = gpx_rows.mean()

stride = tile_size - 1
cmin_gpx = gpx_cols.min()
cmax_gpx = gpx_cols.max()
rmin_gpx = gpx_rows.min()
rmax_gpx = gpx_rows.max()
n_tiles_x = max(1, int(np.ceil((cmax_gpx - cmin_gpx - 1) / stride)))
n_tiles_y = max(1, int(np.ceil((rmax_gpx - rmin_gpx - 1) / stride)))
target_cols = n_tiles_x * stride + 1
target_rows = n_tiles_y * stride + 1

cmin = int(round(centroid_col - (target_cols - 1) / 2))
rmin = int(round(centroid_row - (target_rows - 1) / 2))
cmax = cmin + target_cols
rmax = rmin + target_rows

cmin = max(0, cmin)
rmin = max(0, rmin)
cmax = min(w_out, cmax)
rmax = min(h_out, rmax)
target_cols = cmax - cmin
target_rows = rmax - rmin
n_tiles_x = int(np.ceil((target_cols - 1) / stride))
n_tiles_y = int(np.ceil((target_rows - 1) / stride))
print(f"Target distribution structure: {n_tiles_x}x{n_tiles_y} grid layouts")

# --- 6. CROP, FILL NODATA, NORMALIZE ---
w = Window(cmin, rmin, target_cols, target_rows)
cropped = src_r.read(window=w).astype(np.float32)
crop_tfm = rasterio.windows.transform(w, src_r.transform)

# Fill NoData voids
nodata = src_r.nodata
if nodata is not None:
    mask = cropped[0] != nodata
    cropped[0] = fillnodata(cropped[0], mask)

# Pad to exact grid math if window was truncated at DEM edge
rows, cols = cropped.shape[1], cropped.shape[2]
target_cols_exact = n_tiles_x * stride + 1
target_rows_exact = n_tiles_y * stride + 1
need_pad_c = max(0, target_cols_exact - cols)
need_pad_r = max(0, target_rows_exact - rows)
if need_pad_c > 0 or need_pad_r > 0:
    cropped = np.pad(cropped, ((0, 0), (0, need_pad_r), (0, need_pad_c)), mode='edge')
    rows, cols = cropped.shape[1], cropped.shape[2]

# Normalize to 16-bit
dem_min = float(cropped[0].min())
dem_max = float(cropped[0].max())
dem_16 = ((cropped[0] - dem_min) / (dem_max - dem_min) * 65535).astype(np.uint16)
print(f"Elevation range: {dem_min:.1f}m to {dem_max:.1f}m -> 0-65535")

# Z-Scale calculation for Unreal import
elevation_range = dem_max - dem_min
unreal_z_scale = (elevation_range * 100) / 512
print(f"--> WHEN IMPORTING TO UE5, SET THE PNG 'SCALE X/Y' TO: {100 * pixel_size_m:.4f}")
print(f"--> WHEN IMPORTING TO UE5, SET THE LANDSCAPE 'SCALE Z' TO: {unreal_z_scale:.4f}")

# --- 7. EXPORT HEIGHTMAP & BOUNDS ---

# Export heightmap
hm_dir = os.path.join(output_dir, "heightmaps")
os.makedirs(hm_dir, exist_ok=True)
hm_path = os.path.join(hm_dir, "Heightmap.png")
imageio.v3.imwrite(hm_path, dem_16, plugin="pillow")
print(f"Heightmap exported: {hm_path} ({cols}x{rows} px, {dem_16.nbytes / 1e6:.1f} MB)")

dem_tif_path = os.path.join(output_dir, "reprojected_dem.tif")
with rasterio.open(dem_tif_path, 'w', **src_r.profile) as dst:
    dst.write(src_r.read())
print(f"Reprojected DEM saved: {dem_tif_path}")

x_min = crop_tfm.c
x_max = x_min + cols * abs(crop_tfm.a)
y_max = crop_tfm.f
y_min = y_max - rows * abs(crop_tfm.e)
print(f"PNG bounds (UTM m): x_min={x_min:.1f} x_max={x_max:.1f} y_min={y_min:.1f} y_max={y_max:.1f}")
print(f"Pixel size: {pixel_size_m:.4f} m")

bounds_meta = {
    "x_min": x_min, "y_min": y_min,
    "x_max": x_max, "y_max": y_max,
    "pixel_size_m": pixel_size_m
}
meta_path = os.path.join(output_dir, "heightmap_bounds.json")
with open(meta_path, 'w') as f:
    json.dump(bounds_meta, f, indent=2)
print(f"Bounds metadata saved: {meta_path}")



## GPX Track Conversion

Converts GPX track points into Unreal Engine world coordinates (centimeters) for a spline-based track blueprint. Reads the heightmap bounds to compute the map center, then offsets and flips each point's UTM coordinates into UE5 space.

In [ ]:
# --- PARAMETERS ---
gpx_path = "../data/wurl/WURL_Wasatch_Ultimate_Ridge_Linkup.gpx"
bounds_path = "../data/wurl/unreal_output/heightmap_bounds.json"
output_dir = "../data/wurl/unreal_output"
track_z_cm = 150000.0  # well above terrain for Blueprint shrink-wrapping

# --- 1. LOAD GPX & DETERMINE UTM ZONE ---
with open(gpx_path) as f:
    gpx = gpxpy.parse(f)
points = []
for track in gpx.tracks:
    for segment in track.segments:
        for pt in segment.points:
            points.append({'lat': pt.latitude, 'lon': pt.longitude, 'ele': pt.elevation or 0})
if not points:
    raise ValueError("No track points found in GPX.")
print(f"Loaded GPX: {len(points)} points")

center_lon = np.mean([p['lon'] for p in points])
center_lat = np.mean([p['lat'] for p in points])
zone = int(np.floor((center_lon + 180) / 6)) + 1
hemisphere = 6 if center_lat >= 0 else 7
target_crs = f"EPSG:326{zone:02d}" if hemisphere == 6 else f"EPSG:327{zone:02d}"
print(f"UTM zone: {zone}{'N' if hemisphere == 6 else 'S'}  ({target_crs})")

proj = pyproj.Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)
coords = np.array([proj.transform(p['lon'], p['lat']) for p in points])

# --- 2. LOAD HEIGHTMAP BOUNDS ---
with open(bounds_path) as f:
    b = json.load(f)
x_min, y_min = b["x_min"], b["y_min"]
x_max, y_max = b["x_max"], b["y_max"]
pixel_size_m = b["pixel_size_m"]

cols = int(round((x_max - x_min) / pixel_size_m))
rows = int(round((y_max - y_min) / pixel_size_m))
print(f"Heightmap grid: {cols}x{rows} px, {pixel_size_m:.4f} m/px")

# --- 3. CONVERT TO UNREAL WORLD COORDINATES (cm) ---
# Map center in UTM meters
map_center_easting = (x_min + x_max) / 2.0
map_center_northing = (y_min + y_max) / 2.0

# Offset from center, flip Y (UTM north-up -> UE5 south-up), scale to cm
x_unreal = (coords[:, 0] - map_center_easting) * 100
y_unreal = -(coords[:, 1] - map_center_northing) * 100
z_unreal = np.full_like(x_unreal, track_z_cm)

# --- 4. EXPORT CSV ---
df = pd.DataFrame({
    'Point_ID': range(len(x_unreal)),
    'X': x_unreal,
    'Y': y_unreal,
    'Z': z_unreal
})

csv_path = os.path.join(output_dir, "track_points.csv")
df.to_csv(csv_path, index=False)
print(f"CSV exported: {csv_path} ({len(df)} points)")
